In [ ]:
from __future__ import annotations
import pandas as pd
from pathlib import Path
import copy

from collections import defaultdict
from typing import Any, Dict, Iterable, List, Optional, Union
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import seaborn as sns
sns.set_style('whitegrid')

#switch dir and load packages
import os

os.chdir(r'..\..')

from app.dike_components.dike_model import DikeModel
from app.cost_calculator import CostCalculator, DirectCostGroundWork, StructureCosts, InfrastructureCosts
from app.unit_costs_and_surcharges import load_kosten_catalogus
from dev.incremental_costs.utils import get_dimensions_dict_from_df, modified_cost_computation, make_reinforcement_incremental, compute_incremental_volumes, compute_incremental_costs, compute_lcc, lcc_plot, recategorize_cost



Dit notebook geeft een voorbeeld hoe kosten kunnen worden bepaald uit output files van de Verkenning-2.0-backend
We gebruiken 3 kosteninschattingen van de Scheldestromen casus als voorbeeld.

## Invoerpaden en incrementele versterkingen
In onderstaande cel moet de werkmap met de kostenuitvoer worden geselecteerd, en de filenamen van de verschillende versterkingen. Daarbij kan per file een naam van de maatregel worden opgegeven waarmee later evt. kan worden gecombineerd tot adaptatiepad.

In [ ]:
# Path of WJK

working_dir = Path(r"c:\Users\klerk_wj\Stichting Deltares\KIA – Aanpasbaar en Uitbreidbaar - Documents\WP3 casestudies\2c WIP casus WSSS\designs toolbox KOSWAT")
results_dir = working_dir.parent.joinpath("results")

# Path of Yida
# working_dir = Path(r"C:\Users\tao\OneDrive - Stichting Deltares\KIA – Aanpasbaar en Uitbreidbaar - Documents\WP3 casestudies\2c WIP casus WSSS\designs toolbox KOSWAT")

sns.set_palette("husl", 12)
colors = sns.color_palette("husl", 12)



## Berekenen van kosten van incrementele versterkingen
Nu gaan we de kosten van incrementele versterkingen bepalen. Deze worden gebaseerd op de met de toolbox bepaalde kosten. 
We hebben dus kosten van bijv huidige situatie naar Grondversterking 2026 en huidige situatie naar Grondversterking 2075.
We gaan nu de kosten berekenen voor Grondversterking 2025 naar Grondversterking 2075, dus alleen de meerkosten.

## Adaptatiepad 1: grondversterking
We kijken eerst puur naar het adaptatiepad voor grondversterking.

In [ ]:
## SETTINGS ## 
files = ['ontwerpvariant grond gestacked/kostenoverzicht design grond 1.xlsx',
         'ontwerpvariant grond gestacked/kostenoverzicht grond 2 aanlegprofiel.xlsx',
         'ontwerpvariant grond gestacked/kostenoverzicht grond 3 aanlegprofiel.xlsx',]

names = ["Grondversterking 2025", "Grondversterking 2075", "Grondversterking 2125"]

alternatives = dict(zip(names, files))

dike = DikeModel(complexity = 'gemiddelde maatregel')

#load costs for all alternatives
cost_dataframes_per_alternative = {}
for name, file in alternatives.items():
    cost_dataframes_per_alternative[name] = pd.read_excel(working_dir / file, sheet_name='Kosten')

#dimensions for all alternatives
dimensions = {}
for name, df in cost_dataframes_per_alternative.items():
    dimensions[name] = get_dimensions_dict_from_df(df)

#reduce infrastructure dimensions for 2075 and 2125 to 2000 as we assume it only concerns the road:
dimensions["Grondversterking 2075"]["infrastructure"]['Weg'] = 2000
dimensions["Grondversterking 2125"]["infrastructure"]['Weg'] = 2000 
#recomputation of plain costs for all alternatives
cost_summaries = {}
full_cost_dicts = {}
new_dimensions = {}
for name, dimension in dimensions.items():
    cost_summaries[name], full_cost_dicts[name], new_dimensions[name] = modified_cost_computation(dike, dimension)
    print(f"Kosten van beginsituatie naar {name}:")
    print(f"€{cost_summaries[name]['Kosten excl. BTW'].sum():,.0f}")

df_all_costs = pd.concat(cost_summaries.values(), axis=1)
df_all_costs.columns = cost_summaries.keys()
print("Kosten van alle losse alternatieven:")
#add (ruw) to the column names of df_all_costs
df_all_costs.columns = [col + " (ruw)" for col in df_all_costs.columns]

cost_per_increment_summary, cost_per_increment_detailed, dimensions_per_increment = compute_incremental_costs(names, dimensions, dike)

df_all_increments = pd.concat(cost_per_increment_summary.values(), axis=1)
df_all_increments.columns = cost_per_increment_summary.keys()
df_all_increments.columns = [col + " (incrementeel)" for col in df_all_increments.columns]


#merge the incremental costs with the plain costs
df_all_costs_with_increments = pd.concat([df_all_costs, df_all_increments], axis=1)
df_all_costs_with_increments
#add (incrementeel) to the column names of df_all_increments before merging

#add Vastgoedkosten 1300000 for Grondversterking 2025
df_all_costs_with_increments.loc['Vastgoedkosten', 'Grondversterking 2025 (ruw)'] = 1300000
df_all_costs_with_increments.loc['Vastgoedkosten', 'Grondversterking 2075 (ruw)'] = 1300000 + 100 * (250*3)
df_all_costs_with_increments.loc['Vastgoedkosten', 'Grondversterking 2125 (ruw)'] = 1300000 + 100 * (250*3) + 100 * (250*5)

#add grondaankoop kosten for Grondversterking 2075 and 2125:
df_all_costs_with_increments.loc['Vastgoedkosten', 'Grondversterking 2025 (incrementeel)'] = 1300000
df_all_costs_with_increments.loc['Vastgoedkosten', 'Grondversterking 2025 to Grondversterking 2075 (incrementeel)'] = 100 * (250*3)
df_all_costs_with_increments.loc['Vastgoedkosten', 'Grondversterking 2075 to Grondversterking 2125 (incrementeel)'] = 100 * (250*5)

print("Kosten per versterking (ruw en incrementeel):")
df_all_costs_with_increments


In [ ]:
df_all_costs_with_increments = recategorize_cost(df_all_costs_with_increments)
df_all_costs_with_increments

In [ ]:
#make a stacked bar plot of all the increment costs per measure
fig, ax = plt.subplots(figsize=(6,4))
increment_costs_only = df_all_costs_with_increments.loc[:, [col for col in df_all_costs_with_increments.columns if '(incrementeel)' in col]]
increment_costs_only.T.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Incrementele kosten per maatregel')
ax.set_ylabel('Kosten (M€)')
ax.set_ylim(0, np.ceil(ax.get_ylim()[1] / 1e6)*1e6)
#make legend outside of the plot
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, bbox_to_anchor=(1.05, 1), loc='upper left')


#scale y-axislabels to millions
ax.set_yticklabels([f"{y/1e6:.1f}" for y in ax.get_yticks()])

#rotate x-axis labels and remove (incrementeel) and everything before "to"from the labels
ax.set_xticklabels([label.get_text().split(' to ')[-1].replace(' (incrementeel)', '') for label in ax.get_xticklabels()], rotation=0, ha='center')
# ax.set_ylim(0, ax.get_ylim()[1] / 1e6)


In [ ]:
#same plot for (ruw) costs
fig, ax = plt.subplots(figsize=(6,4))
raw_costs_only = df_all_costs_with_increments.loc[:, [col for col in df_all_costs_with_increments.columns if '(ruw)' in col]]
raw_costs_only.T.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Ruwe kosten per maatregel')
ax.set_ylabel('Kosten (M€)')
ax.set_ylim(0, np.ceil(ax.get_ylim()[1] / 1e6)*1e6)
#make legend outside of the plot
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, bbox_to_anchor=(1.05, 1), loc='upper left')

#scale y-axislabels to millions
ax.set_yticklabels([f"{y/1e6:.1f}" for y in ax.get_yticks()])
#rotate x-axis labels and remove (ruw) and everything before "to" from the labels
ax.set_xticklabels([label.get_text().split(' to ')[-1].replace(' (ruw)', '') for label in ax.get_xticklabels()], rotation=0, ha='center')


In [ ]:
#LCC computation
#make dict with sum of incremental costs per measure, year, lifespan (50 for soil) in a tuple
incremental_costs_per_measure = {}
for col in increment_costs_only.columns:
    measure_name = col.split(' (incrementeel)')[0]
    #find the year in the measure name 
    year = int(measure_name.split(' ')[-1])
    incremental_costs_per_measure[measure_name] = (increment_costs_only[col].sum(), year, 100)
    # incremental_costs_per_measure[measure_name] = (increment_costs_only[col].sum(), year, 50)

total_horizon = 150
lcc_per_measure = compute_lcc(incremental_costs_per_measure, total_horizon=total_horizon)
lcc_per_measure

undiscounted_costs_adaptation_path_1 = [cost for cost, year, lifespan in incremental_costs_per_measure.values()]
lcc_values_adaptation_path_1 = [lcc for lcc in lcc_per_measure.values()]
years_adaptation_path_1 = [year for cost, year, lifespan in incremental_costs_per_measure.values()]



lcc_plot(undiscounted_costs_adaptation_path_1, lcc_values_adaptation_path_1, years_adaptation_path_1, total_horizon, title='Levenscycluskosten voor Adaptatiepad 1 - Grondversterking')

### Gevoeligheid snelheid ZSS


In [ ]:
from scipy.interpolate import interp1d
ZSS_HBN_relatie = interp1d([7.952, 10.40], [0, 2])

# ZSS_per_increment = {'Grondversterking 2025': 0.0, 'Grondversterking 2075': 0.5, 'Grondversterking 2125': 1.0625, 'Grondversterking 2175': 1.5}
# ZSS_per_increment = {'Grondversterking 2025': 0.57, 'Grondversterking 2075': 1.1325, 'Grondversterking 2125': 1.71}
HBN_per_increment = {'Grondversterking 2025': 8.87, 'Grondversterking 2075': 9.25, 'Grondversterking 2125': 10.02}
#for a given ZSS per year, compute what year the ZSS_per_increment are exceeded and return the exceedences of each

def get_exceedance_years(HBN_per_increment, ZSS_HBN_relatie, ZSS_per_year):
    exceedance_years = {}
    for measure, HBN in HBN_per_increment.items():
        ZSS = ZSS_HBN_relatie(HBN)
        year_exceeded = math.ceil(ZSS / ZSS_per_year)
        exceedance_years[measure] = year_exceeded

    # #translate to lifespan of the measure
    # for measure, year_exceeded in exceedance_years.items():
    return exceedance_years

#compute cost per discounted service year
def get_discounted_service_years(duration_of_service):
    discount_rate_until_35 = 0.022
    discount_rate_after_35 = 0.014
    #determine the discount factorsfor a horizon of 150 years with a step of 1 years, using the discount rate until 35 years and the discount rate after 35 years (also compute for extra years to be sure)
    discount_factors = [(1+ discount_rate_until_35) ** t if t <= 35 else (1+ discount_rate_after_35) ** (t-35) * (1+ discount_rate_until_35) ** 35 for t in range(0, 501, 1)]

    discounted_service_years = [1/df for df in discount_factors[1:duration_of_service+1]]  # skip the first year (t=0)
    return np.sum(discounted_service_years)

ZSS_per_year = 0.011428571 #m/year
ZSS_rates = [ZSS_per_year * 0.5, ZSS_per_year, ZSS_per_year * 1.5] #50% slower, normal, 50% faster
exceedance_years = []
LCC_per_service_year = []

for ZSS_rate in ZSS_rates:
    print(f"ZSS per year: {ZSS_rate} m/year")
    exceedance_years.append(get_exceedance_years(HBN_per_increment, ZSS_HBN_relatie, ZSS_rate))
    print(f"Exceedance years for ZSS rate {ZSS_rate}: {exceedance_years[-1]}")
    investment_years = [year + 2025 for year in exceedance_years[-1].values()]
    #service horizons are years, but we need to compute the lifespan so the differences.
    service_horizons = [investment_years[0]] + [investment_years[i] - investment_years[i-1] for i in range(1, len(investment_years))]
    #compute LCC for the computed exceedance years
    #first copy and modify the incremental_costs_per_measure to have the new service horizons
    incremental_costs_per_measure_modified = {}
    for ind, (measure, cost) in enumerate(incremental_costs_per_measure.items()):
        incremental_costs_per_measure_modified[measure] = (cost[0], investment_years[ind], service_horizons[ind])

    lcc_scenario = compute_lcc(incremental_costs_per_measure_modified, total_horizon=500)
    print(f"LCC for ZSS rate {ZSS_rate}: {lcc_scenario}")
    discounted_service_years = get_discounted_service_years(max(service_horizons))
    print(f"Discounted service years for ZSS rate {ZSS_rate}: {discounted_service_years:.2f}")
    lcc_per_service_year = sum(list(lcc_scenario.values())) / discounted_service_years
    print(f"LCC per discounted service year for ZSS rate {ZSS_rate}: {lcc_per_service_year:.2f}")
    LCC_per_service_year.append(lcc_per_service_year)


#plot
fig, ax = plt.subplots(figsize=(6,4))
x = ['50% slower', 'Expected', '50% faster']
ax.bar(x, LCC_per_service_year)
ax.set_ylabel('LCC per discounted service year')
ax.set_title('LCC per discounted service year for different ZSS rates')
plt.show()



## Adaptatiepad 2 - constructief

Maatregelen, en welk grondprofiel er bij hoort:
2025:
* Verankerde damwand 2025

2075:
* 2a: Oplassen & ophoging kruin + taludverflauwing 2075
* 2b: Ophogen kruin + taludverflauwing + plaatsen L-wanden 2075
* nvt, geen grond: L-wand/keermuur op kruin

2125:
* 3a: Ophogen kruin + oplassen + buispalen bijplaatsen 2125
* 3b: Ophogen kruin + plaatsen L-wand
* 2d: Ophogen kruin, kistdam in binnenteen
* 2d: Ophogen kruin + kistdam in kruin + afschrijven damwand
* grond 2125: Binnenwaarts in grond, damwand afschrijven


In [ ]:
list(zip(files,names))

In [ ]:
## SETTINGS ## 
files = ['ontwerpvarianten constructief/kostenoverzicht_damwand15m_verankerd.xlsx',
         'ontwerpvarianten constructief/kostenoverzicht 2a.xlsx',
         'ontwerpvarianten constructief/kostenoverzicht 2b.xlsx',
         'ontwerpvarianten constructief/kostenoverzicht_damwand15m_verankerd.xlsx',
         'ontwerpvarianten constructief/kostenoverzicht 3a.xlsx',
         'ontwerpvarianten constructief/kostenoverzicht 3b.xlsx',
         'ontwerpvarianten constructief/kostenoverzicht 3d.xlsx',
         'ontwerpvarianten constructief/kostenoverzicht 3d.xlsx',
         'ontwerpvariant grond gestacked/kostenoverzicht grond 3 aanlegprofiel.xlsx',
         #vanaf hier is tbv van gevoeligheidsanalyse
         'ontwerpvariant grond gestacked/kostenoverzicht geen_aanvulling.xlsx',
         'ontwerpvarianten constructief/kostenoverzicht 2a.xlsx',
         'ontwerpvarianten constructief/kostenoverzicht 3d.xlsx',
         ]

names = ["Verankerde damwand 2025",
         "Ophogen kruin + Taludverflauwing + Oplassen damwand 2075",
         "Ophogen kruin + Taludverflauwing + Plaatsen L-wanden 2075",
         "L-wand op kruin 2075",
         "Ophogen kruin + Oplassen + Buispalen bijplaatsen 2125",
         "Ophogen kruin + Plaatsen L-wanden 2125",
         "Ophogen kruin + Kistdam in binnenteen 2125",
         "Ophogen kruin + Kistdam in kruin + Afschrijven damwand 2125",
         "Grondversterking 2125",
         "Kistdam in teen zonder grondaanvulling 2075",
         "Kistdam in teen met grondaanvulling 2075",
         "Grondaanvulling bij bestaande kistdam 2125"
         ]

#  unit prices for custom structures #TODO check and align values:
unit_prices = {'oplassen variabel per meter damwand per strekkende meter': 300, #o.b.v. prijzen korte damwand (zwaarder profiel). 
               'oplassen vaste kosten per strekkende meter': 280 * 1.5, #o.b.v. unit price stalen gording UNP240. Keer 1.5 voor extra complexiteit
               'L-wand 1.5 meter hoog per strekkende meter': 200, #o.b.v. kosten bij agri-beton.nl (keerwand 150x200x78)
               'Buispalen bijplaatsen per strekkende meter': (7000 / 2.8),  # EU/st / h.o.h. 7000 per buispaal, 2.8m per buispaal
               'Kistdam in binnenteen per strekkende meter':6000, #   Lev/aanbr. kistdam, AZ36, inheid. 15m (B=4m) kost 10000, daar halen we de bestaande damwand vanaf (4000)
               'Kistdam in kruin per strekkende meter': 13000, #  Lev/aanbr. kistdam, AZ36, inheid. 20m (B=4m)
               'Afbranden bestaande damwand per strekkende meter': 200,  #nog checken,
                }

 
alternatives = dict(zip(names, files))

dike = DikeModel(complexity = 'gemiddelde maatregel')

#load costs for all alternatives
cost_dataframes_per_alternative = {}
for name, file in alternatives.items():
    cost_dataframes_per_alternative[name] = pd.read_excel(working_dir / file, sheet_name='Kosten')

#dimensions for all alternatives
dimensions = {}
for name, df in cost_dataframes_per_alternative.items():
    dimensions[name] = get_dimensions_dict_from_df(df)

vaklengte = dimensions['Verankerde damwand 2025']['structure']['Vaklengte'].copy()
# #update costs to include the custom structure costs.
for name, dimension in dimensions.items():
    if name == "Ophogen kruin + Taludverflauwing + Oplassen damwand 2075":
        lengte_oplassen = 3.5 #m, aannemend dat er 1 vak van 3.5m nodig is om de kruin te versterken, dit is mogelijk optimistisch
        dimension['structure'] = {'Type': f'Oplassen damwand {lengte_oplassen} meter', 
                                  'Vaklengte': vaklengte, 
                                  'Eenheidsprijs': unit_prices['oplassen variabel per meter damwand per strekkende meter'] * lengte_oplassen + unit_prices['oplassen vaste kosten per strekkende meter']}
    elif name == "Ophogen kruin + Taludverflauwing + Plaatsen L-wanden 2075":
        aantal_wanden = 2
        dimension['structure'] = {'Type': f'Plaatsen {aantal_wanden} L-wanden bij teen', 
                                  'Vaklengte': vaklengte, 
                                  'Eenheidsprijs': unit_prices['L-wand 1.5 meter hoog per strekkende meter'] * aantal_wanden}
        pass
    elif name == "L-wand op kruin 2075":
        dimension['structure'] = {'Type': f'L-wand 1.5 meter hoog op kruin', 
                                  'Vaklengte': vaklengte,
                                    'Eenheidsprijs': unit_prices['L-wand 1.5 meter hoog per strekkende meter']}
    elif name == "Ophogen kruin + Oplassen + Buispalen bijplaatsen 2125":
        lengte_oplassen = 1.8
        dimension['structure'] = {'Type': f'Oplassen damwand {lengte_oplassen} meter + Buispalen bijplaatsen',
                                    'Vaklengte': vaklengte,
                                    'Eenheidsprijs': unit_prices['oplassen variabel per meter damwand per strekkende meter'] * lengte_oplassen + unit_prices['oplassen vaste kosten per strekkende meter'] + unit_prices['Buispalen bijplaatsen per strekkende meter']}
    elif name == "Ophogen kruin + Plaatsen L-wanden 2125":
        aantal_wanden = 5
        dimension['structure'] = {'Type': f'Plaatsen {aantal_wanden} L-wanden bij teen',
                                    'Vaklengte': vaklengte,
                                    'Eenheidsprijs': unit_prices['L-wand 1.5 meter hoog per strekkende meter'] * aantal_wanden}
    elif name == "Ophogen kruin + Kistdam in binnenteen 2125":
        dimension['structure'] = {'Type': f'Kistdam in binnenteen',
                                    'Vaklengte': vaklengte,
                                    'Eenheidsprijs': unit_prices['Kistdam in binnenteen per strekkende meter']}
    elif name == "Ophogen kruin + Kistdam in kruin + Afschrijven damwand 2125":
        dimension['structure'] = {'Type': f'Kistdam in kruin + Afschrijven damwand',
                                    'Vaklengte': vaklengte,
                                    'Eenheidsprijs': unit_prices['Kistdam in kruin per strekkende meter'] + unit_prices['Afbranden bestaande damwand per strekkende meter']}
    elif name == "Grondversterking 2125":
        dimension['structure'] = {'Type': f'Afbranden bestaande damwand',
                                    'Vaklengte': vaklengte,
                                    'Eenheidsprijs': unit_prices['Afbranden bestaande damwand per strekkende meter']}
    elif name in ["Kistdam in teen zonder grondaanvulling 2075", "Kistdam in teen met grondaanvulling 2075"]:
        dimension['structure'] = {'Type': f'Kistdam in binnenteen',
                                    'Vaklengte': vaklengte,
                                    'Eenheidsprijs': unit_prices['Kistdam in binnenteen per strekkende meter']}
    elif name == "Grondaanvulling bij bestaande kistdam 2125":
        lengte_oplassen = 1.5
        dimension['structure'] = {'Type': f'Grondaanvulling bij bestaande kistdam',
                                    'Vaklengte': vaklengte,
                                    'Eenheidsprijs': unit_prices['oplassen variabel per meter damwand per strekkende meter'] * lengte_oplassen + unit_prices['oplassen vaste kosten per strekkende meter']} #nog checken, aannemend dat dit een relatief goedkope maatregel is
    else:
        pass

#recomputation of plain costs for all alternatives
cost_summaries = {}
full_cost_dicts = {}
new_dimensions = {}
for name, dimension in dimensions.items():
    cost_summaries[name], full_cost_dicts[name], new_dimensions[name] = modified_cost_computation(dike, dimension)
    print(f"Kosten van beginsituatie naar {name}: €{cost_summaries[name]['Kosten excl. BTW'].sum():,.0f}")

df_all_costs = pd.concat(cost_summaries.values(), axis=1)
df_all_costs.columns = cost_summaries.keys()
#add (ruw) to the column names of df_all_costs
df_all_costs.columns = [col + " (ruw)" for col in df_all_costs.columns]

df_all_costs.loc['Vastgoedkosten', 'Grondversterking 2125 (ruw)'] = 1300000 + 100 * (250*3) + 100 * (250*5)




In [ ]:
df_all_costs

In [ ]:
#same plot for (ruw) costs
fig, ax = plt.subplots(figsize=(12,4))
df_all_costs.T.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Ruwe kosten per maatregel')
ax.set_ylabel('Kosten (M€)')
ax.set_ylim(0, np.ceil(ax.get_ylim()[1] / 1e6)*1e6)
#make legend outside of the plot
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, bbox_to_anchor=(1.05, 1), loc='upper left')

#scale y-axislabels to millions
ax.set_yticklabels([f"{y/1e6:.1f}" for y in ax.get_yticks()])
#rotate x-axis labels and remove (ruw) and everything before "to" from the labels
ax.set_xticklabels([label.get_text().split(' to ')[-1].replace(' (ruw)', '') for label in ax.get_xticklabels()], rotation=45, ha='right')

### Adaptatiepad 2: constructief

In [ ]:
measure_orders = {"Adaptatiepad 2a - damwand met oplassen": ["Verankerde damwand 2025", "Ophogen kruin + Taludverflauwing + Oplassen damwand 2075", "Ophogen kruin + Oplassen + Buispalen bijplaatsen 2125",],
                  "Adaptatiepad 2b - damwand met L-wanden": ["Verankerde damwand 2025", "Ophogen kruin + Taludverflauwing + Plaatsen L-wanden 2075", "Ophogen kruin + Plaatsen L-wanden 2125",],
                  "Adaptatiepad 2c - damwand met kruinmuur & kistdam in teen": ["Verankerde damwand 2025", "L-wand op kruin 2075", "Ophogen kruin + Kistdam in binnenteen 2125",],
                  "Adaptatiepad 2d - damwand met kruinmuur & kistdam in kruin": ["Verankerde damwand 2025", "L-wand op kruin 2075", "Ophogen kruin + Kistdam in kruin + Afschrijven damwand 2125",],
                  "Adaptatiepad 2e - damwand met kruinmuur & grondversterking": ["Verankerde damwand 2025", "L-wand op kruin 2075", "Grondversterking 2125",],
                #   "Adaptatiepad 2f - damwand met oplassen & kistdam in teen": ["Verankerde damwand 2025", "Ophogen kruin + Taludverflauwing + Oplassen damwand 2075", "Ophogen kruin + Kistdam in binnenteen 2125",],
                  }

adaptation_paths_summary, adaptation_paths_detailed, adaptation_paths_dimensions = {}, {}, {}
#compute incremental costs for the different measure orders
for adaptation_path, measure_order in measure_orders.items():
    adaptation_paths_summary[adaptation_path], adaptation_paths_detailed[adaptation_path], adaptation_paths_dimensions[adaptation_path] = compute_incremental_costs(measure_order, dimensions, dike)
    summary, adaptation_paths_detailed[adaptation_path], adaptation_paths_dimensions[adaptation_path] = compute_incremental_costs(measure_order, dimensions, dike)

    adaptation_paths_summary[adaptation_path] = pd.concat(summary.values(), axis=1)
    adaptation_paths_summary[adaptation_path].columns = summary.keys()

    if 'Grondversterking 2125' in measure_order:
        #add Vastgoedkosten for measure that contains "to Grondversterking 2125"
        adaptation_paths_summary[adaptation_path].loc['Vastgoedkosten', 
                                                      adaptation_paths_summary[adaptation_path].columns[adaptation_paths_summary[adaptation_path].columns.str.contains('to Grondversterking 2125')]] = 1300000 + 100 * (250*5)


In [ ]:
#make a stacked bar plot of all the increment costs per adaptation path. We have 5 paths, so make 6 subplots (2x3), one for each path and one where we can put the legend
# 

fig, axes = plt.subplots(2, 3, figsize=(18,10))
axes = axes.flatten()

for i, (adaptation_path, summary) in enumerate(adaptation_paths_summary.items()):
    summary.T.plot(kind='bar', stacked=True, ax=axes[i])
    axes[i].set_title(adaptation_path)
    #no legend
    axes[i].legend_.remove()
    axes[i].set_ylabel('Kosten (M€)')
    axes[i].set_ylim(0, 1.2e7)
    #rotate the labels and split along multiple lines if they are too long (max length per line is 15 characters. Split at ' to ' and + if possible, otherwise split at the max length of 15 characters per line)
    #rotate x-axis labels and remove (incrementeel) and everything before "to"from the labels. Replace all " + " with "\n" for better readability. Remove 2025, 2075 and 2125 from the labels for better readability.
    axes[i].set_xticklabels([label.get_text().split(' to ')[-1].replace(' + ', '\n').replace('2025', '').replace('2075', '').replace('2125', '') for label in axes[i].get_xticklabels()], rotation=0, ha='center')
    axes[i].set_yticklabels([f"{y/1e6:.0f}" for y in axes[i].get_yticks()])
    # #add horizontal padding between subplots
plt.subplots_adjust(hspace=.4)

#make space of 6th plot entirely white
axes[5].axis('off')
handles, labels = axes[4].get_legend_handles_labels()
axes[4].legend(handles, labels, bbox_to_anchor=(1.25, 1), loc='upper left')

fig.suptitle('Incrementele kosten per adaptatiepad voor constructieve maatregelen', fontsize=16)


In [ ]:
#LCC computation for each adaptation path
#make dict with sum of incremental costs per measure, year, lifespan (50 for soil) in a tuple
adaptation_paths_lcc = {}
adaptation_paths_investment = {}
adaptation_paths_years = {}
for adaptation_path, summary in adaptation_paths_summary.items():
    incremental_costs_per_measure = {}
    for increment_name in summary.columns:
        #find the year in the measure name
        measure_name = increment_name.split(' to ')[-1]
        year = int(measure_name.split(' ')[-1])
        if 'Verankerde damwand' in measure_name or 'Kistdam' in measure_name or 'Buispalen' in measure_name:
            incremental_costs_per_measure[measure_name] = (summary[increment_name].sum(), year, 100)
        else:
            incremental_costs_per_measure[measure_name] = (summary[increment_name].sum(), year, 50)

    total_horizon = 150
    adaptation_paths_lcc[adaptation_path] = compute_lcc(incremental_costs_per_measure, total_horizon=total_horizon)
    adaptation_paths_investment[adaptation_path] = [cost for cost, year, lifespan in incremental_costs_per_measure.values()]
    adaptation_paths_years[adaptation_path] = [year for cost, year, lifespan in incremental_costs_per_measure.values()]

    
    lcc_plot(adaptation_paths_investment[adaptation_path], adaptation_paths_lcc[adaptation_path].values(), adaptation_paths_years[adaptation_path], total_horizon, title=f'Levenscycluskosten voor {adaptation_path}', y_top_lim = 10e6)
    plt.savefig(results_dir / f'LCC_{adaptation_path}.png', bbox_inches='tight')
    plt.close()

In [ ]:
#for all adaptation paths get the total LCC
initial_investment_dict = {'Adaptatiepad 1 - grondversterking': lcc_per_measure['Grondversterking 2025']}
lcc_dict = {'Adaptatiepad 1 - grondversterking': sum(lcc_values_adaptation_path_1)}
total_investment_dict = {'Adaptatiepad 1 - grondversterking': sum(undiscounted_costs_adaptation_path_1)}


for adaptation_path, lcc_values in adaptation_paths_lcc.items():
    lcc_dict[adaptation_path] = sum(lcc_values.values())
    initial_investment_dict[adaptation_path] = adaptation_paths_investment[adaptation_path][0]
    total_investment_dict[adaptation_path] = sum(adaptation_paths_investment[adaptation_path])

In [ ]:
#for all adaptation paths get the total LCC
initial_investment_dict = {'Adaptatiepad 1 - grondversterking': lcc_per_measure['Grondversterking 2025']}
lcc_dict = {'Adaptatiepad 1 - grondversterking': sum(lcc_values_adaptation_path_1)}
total_investment_dict = {'Adaptatiepad 1 - grondversterking': sum(undiscounted_costs_adaptation_path_1)}


for adaptation_path, lcc_values in adaptation_paths_lcc.items():
    lcc_dict[adaptation_path] = sum(lcc_values.values())
    initial_investment_dict[adaptation_path] = adaptation_paths_investment[adaptation_path][0]
    total_investment_dict[adaptation_path] = sum(adaptation_paths_investment[adaptation_path])

#bar chart of total LCC per adaptation path
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(lcc_dict.keys(), lcc_dict.values(), label = 'Toekomstige investering')
ax.bar(initial_investment_dict.keys(), initial_investment_dict.values(), color='orange', label='Initiële investering')
ax.set_title('Totale levenscycluskosten per adaptatiepad')
ax.set_ylabel('Kosten (M€)')
ax.set_ylim(0, np.ceil(max(lcc_dict.values()) / 1e6)*1e6)
ax.set_yticklabels([f"{y/1e6:.1f}" for y in ax.get_yticks()])
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

#bar chart of total investment per adaptation path
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(total_investment_dict.keys(), total_investment_dict.values(), label = 'Toekomstige investering')
ax.bar(initial_investment_dict.keys(), initial_investment_dict.values(), color='orange', label='Initiële investering')
ax.set_title('Totale investeringskosten per adaptatiepad')
ax.set_ylabel('Kosten (M€)')
ax.set_ylim(0, np.ceil(max(total_investment_dict.values()) / 1e6)*1e6)
ax.set_yticklabels([f"{y/1e6:.1f}" for y in ax.get_yticks()])
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')


In [ ]:
#make plot of LCC of adaptation paths without the investment in 2025
adaptation_paths_lcc_without_initial_investment = {adaptation_path: lcc-initial_investment_dict[adaptation_path] for adaptation_path, lcc in lcc_dict.items()}
adaptation_paths_investment_without_initial_investment = {adaptation_path: investment_cost-initial_investment_dict[adaptation_path] for adaptation_path, investment_cost in total_investment_dict.items()}

fig, (ax1, ax2) = plt.subplots(figsize=(8,4), ncols=2)
ax1.bar(adaptation_paths_lcc_without_initial_investment.keys(), adaptation_paths_lcc_without_initial_investment.values(), color =colors[1])
ax1.set_title('LCC')
ax1.set_ylabel('Kosten (M€)')
ax1.set_ylim(0,2e6)
ax1.set_yticklabels([f"{y/1e6:.1f}" for y in ax1.get_yticks()])
# ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')

ax2.bar(adaptation_paths_investment_without_initial_investment.keys(), adaptation_paths_investment_without_initial_investment.values(), color =colors[3])
ax2.set_title('Investeringskosten')
ax2.set_ylabel('Kosten (M€)')
ax2.set_ylim(0,12e6)
ax2.set_yticklabels([f"{y/1e6:.1f}" for y in ax2.get_yticks()])
# ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')

fig.suptitle('Kosten van vervolgversterkingen per adaptatiepad', fontsize=16)

#make plot of relative difference of LCC of adaptation paths without the investment in 2025 compared to Adaptatiepad 1 - grondversterking without the investment in 2025
adaptation_paths_lcc_without_initial_investment_relative = {adaptation_path: (lcc-initial_investment_dict[adaptation_path])/(lcc_dict['Adaptatiepad 1 - grondversterking']-initial_investment_dict['Adaptatiepad 1 - grondversterking']) for adaptation_path, lcc in lcc_dict.items()}
adaptation_paths_investment_without_initial_investment_relative = {adaptation_path: (investment_cost-initial_investment_dict[adaptation_path])/(total_investment_dict['Adaptatiepad 1 - grondversterking']-initial_investment_dict['Adaptatiepad 1 - grondversterking']) for adaptation_path, investment_cost in total_investment_dict.items()}

fig, (ax1, ax2) = plt.subplots(figsize=(8,4), ncols=2)
ax1.bar(adaptation_paths_lcc_without_initial_investment_relative.keys(), adaptation_paths_lcc_without_initial_investment_relative.values(), color =colors[1])
ax1.set_title('Relatieve LCC')
ax1.set_ylabel('Relatieve kosten')
ax1.set_ylim(0,5)
# ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')
ax2.bar(adaptation_paths_investment_without_initial_investment_relative.keys(), adaptation_paths_investment_without_initial_investment_relative.values(), color =colors[3])
ax2.set_title('Relatieve investeringskosten')
ax2.set_ylabel('Relatieve kosten')
ax2.set_ylim(0,5)
# ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left)
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')

In [ ]:
toekomstige_opties_spijtkosten = pd.DataFrame.from_dict(adaptation_paths_lcc_without_initial_investment, orient='index')

def bereken_verdisconteerde_spijtkosten(referentiekosten, toekomstige_opties):
    #bepaal de verdisconteerde spijtkosten voor een reeks toekomstige opties ten opzichte van een referentieoptie
    verdisconteerde_spijtkosten = pd.DataFrame(index=toekomstige_opties.index, columns=['Spijtkosten'])
    for optie, kosten in toekomstige_opties.iterrows():
        #bereken de verdisconteerde spijtkosten als het verschil tussen de referentiekosten en de toekomstige optie, gedeeld door de referentiekosten
        verdisconteerde_spijtkosten.loc[optie, 'Spijtkosten'] = float(kosten - referentiekosten)
    return verdisconteerde_spijtkosten

verdisconteerde_spijtkosten = bereken_verdisconteerde_spijtkosten(toekomstige_opties_spijtkosten.loc['Adaptatiepad 1 - grondversterking'], 
                                                                     toekomstige_opties_spijtkosten.drop(index= 'Adaptatiepad 1 - grondversterking'))

results_dir = Path(r'c:\Users\klerk_wj\Stichting Deltares\KIA – Aanpasbaar en Uitbreidbaar - Documents\WP3 casestudies\2c WIP casus WSSS\figuren_sensitivity_analysis')

#load spitjkosten csv
spijt_scenarios = pd.read_csv(results_dir / 'spijtkosten_toekomstige_investeringen.csv', index_col=0)
#plot verdisconteerde spijtkosten links van de basis analyse, rechts van de gevoeligheidsanalyse.
heatmap_100_jaar = spijt_scenarios.reset_index().pivot(index='Levensduur', columns='ZSS snelheid', values='Regret per 100 jaar')
fig, ax = plt.subplots(figsize=(8,4), ncols=2)

# Use one colormap + normalization for both subplots so colors are comparable.
cmap = plt.get_cmap("Reds")
norm = plt.Normalize(vmin=0, vmax=2)

sns.barplot(x=verdisconteerde_spijtkosten.index, y=verdisconteerde_spijtkosten['Spijtkosten']/1e6, ax=ax[0], color="lightgray")
bar_values_meur = (verdisconteerde_spijtkosten['Spijtkosten'] / 1e6).to_numpy()
for patch, value in zip(ax[0].patches, bar_values_meur):
    patch.set_facecolor(cmap(norm(value)))

ax[0].set_xticklabels([label.get_text().replace(' damwand met', '') for label in ax[0].get_xticklabels()], rotation=45, ha='right')
ax[0].set_yticklabels([f"{y:.1f}" for y in ax[0].get_yticks()])
ax[0].set_ylabel('Spijtkosten (M€)')
ax[0].set_xlabel('')
ax[1].set_xlabel('ZSS (cm/jaar)')
ax[1].set_ylabel('Levensduur damwand (jaar)')

ax[0].set_title('Spijtkosten o.b.v. adaptatiepaden')
ax[1].set_title('Spijtkosten o.b.v. gevoeligheidsanalyse')

sns.heatmap(heatmap_100_jaar/1e6, annot=True, fmt=".2f", cmap=cmap, norm=norm, ax=ax[1])

In [ ]:
fig, ax = plt.subplots(figsize=(8,4), ncols=2)
rel_kostentoename = (toekomstige_opties_spijtkosten.drop(index='Adaptatiepad 1 - grondversterking') / toekomstige_opties_spijtkosten.loc['Adaptatiepad 1 - grondversterking']) - 1

# Use one colormap + normalization for both subplots so colors are comparable.
cmap = plt.get_cmap("Reds")
norm = plt.Normalize(vmin=0, vmax=100)

sns.barplot(x=rel_kostentoename.index, y=rel_kostentoename[0], ax=ax[0], color="lightgray")
bar_values_pct = rel_kostentoename[0].to_numpy() * 100
for patch, value in zip(ax[0].patches, bar_values_pct):
    patch.set_facecolor(cmap(norm(value)))

ax[0].set_xticklabels([label.get_text().replace(' damwand met', '') for label in ax[0].get_xticklabels()], rotation=45, ha='right')
ax[0].set_yticklabels([f"{y*100:.0f}%" for y in ax[0].get_yticks()])
ax[0].set_ylabel('Relatieve extra kosten (%)')

rel_extra_kosten_sensitivity = spijt_scenarios.reset_index().pivot(index='Levensduur', columns='ZSS snelheid', values='Relatieve kostentoename')
rel_extra_kosten_sensitivity = (rel_extra_kosten_sensitivity - 1) * 100
sns.heatmap(data=rel_extra_kosten_sensitivity, annot=True, fmt=".0f", cmap=cmap, norm=norm, ax=ax[1])
ax[1].set_xlabel('ZSS (cm/jaar)')
#change colorbar label to "Relatieve extra kosten (%)"
ax[1].collections[0].colorbar.set_label('Relatieve extra kosten (%)')
#add % to annotations
for t in ax[1].texts:
    t.set_text(t.get_text() + '%')
ax[0].set_title('Relatieve extra kosten\n o.b.v. adaptatiepaden')
ax[1].set_title('Relatieve extra kosten\n o.b.v. gevoeligheidsanalyse')
ax[0].set_xlabel('')

In [ ]:
# Combined figure with only the ax[0] bar plots from the two cells above.
fig, ax = plt.subplots(figsize=(8,6), ncols=2)


# Right panel: discounted regret costs (from the second cell above).
cmap_right = plt.get_cmap("RdYlGn_r")
norm_right = plt.Normalize(vmin=-2, vmax=2)

sns.barplot(x=verdisconteerde_spijtkosten.index, y=verdisconteerde_spijtkosten['Spijtkosten'] / 1e6, ax=ax[0], color="lightgray")
bar_values_meur = (verdisconteerde_spijtkosten['Spijtkosten'] / 1e6).to_numpy()
for patch, value in zip(ax[0].patches, bar_values_meur):
    patch.set_facecolor(cmap_right(norm_right(value)))

ax[0].set_xticklabels([label.get_text().replace(' damwand met', '') for label in ax[0].get_xticklabels()], rotation=45, ha='right')
ax[0].set_yticklabels([f"{y:.1f}" for y in ax[0].get_yticks()])
ax[0].set_ylabel('Spijtkosten (M€)')
ax[0].set_title('Spijtkosten o.b.v. adaptatiepaden')
ax[0].set_xlabel('')
# Left panel: relative extra costs (from the nearest cell above).
cmap_left = plt.get_cmap("Reds")
norm_left = plt.Normalize(vmin=0, vmax=100)

sns.barplot(x=rel_kostentoename.index, y=rel_kostentoename[0], ax=ax[1], color="lightgray")
bar_values_pct = rel_kostentoename[0].to_numpy() * 100
for patch, value in zip(ax[1].patches, bar_values_pct):
    patch.set_facecolor(cmap_left(norm_left(value)))

ax[1].set_xticklabels([label.get_text().replace(' damwand met', '') for label in ax[1].get_xticklabels()], rotation=45, ha='right')
ax[1].set_yticklabels([f"{y*100:.0f}%" for y in ax[1].get_yticks()])
ax[1].set_ylabel('Relatieve extra kosten (%)')
ax[1].set_title('Relatieve extra kosten\n o.b.v. adaptatiepaden')
ax[1].set_xlabel('')


plt.tight_layout()

In [ ]:
# Combined figure with the ax[1] heatmaps from the two cells above.
fig, ax = plt.subplots(figsize=(10, 4), ncols=2)

# Left panel: regret costs heatmap (M€).
cmap_left = plt.get_cmap("Reds")
norm_left = plt.Normalize(vmin=0, vmax=2)

sns.heatmap(
    heatmap_100_jaar / 1e6,
    annot=True,
    fmt=".2f",
    cmap=cmap_left,
    norm=norm_left,
    ax=ax[0]
)
ax[0].set_title('Spijtkosten (M€)')
ax[0].set_xlabel('ZSS (cm/jaar)')
ax[0].set_ylabel('Levensduur damwand (jaar)')
ax[0].collections[0].colorbar.set_label('Spijtkosten (M€)')

# Right panel: relative extra costs heatmap (%).
rel_extra_kosten_sensitivity = spijt_scenarios.reset_index().pivot(
    index='Levensduur',
    columns='ZSS snelheid',
    values='Relatieve kostentoename'
)
rel_extra_kosten_sensitivity = (rel_extra_kosten_sensitivity - 1) * 100

cmap_right = plt.get_cmap("Reds")
norm_right = plt.Normalize(vmin=0, vmax=100)

sns.heatmap(
    data=rel_extra_kosten_sensitivity,
    annot=True,
    fmt=".0f",
    cmap=cmap_right,
    norm=norm_right,
    ax=ax[1]
)
ax[1].set_title('Relatieve extra kosten o.b.v. gevoeligheidsanalyse')
ax[1].set_xlabel('ZSS (cm/jaar)')
ax[1].set_ylabel('Levensduur damwand (jaar)')
ax[1].collections[0].colorbar.set_label('Relatieve extra kosten (%)')

# Add percent signs to annotation values in the relative-cost heatmap.
for t in ax[1].texts:
    t.set_text(t.get_text() + '%')

plt.tight_layout()